# MIMII EDA — Fan & Slider (0 dB)

Load 5 normal + 5 abnormal samples per machine type, plot waveforms and log-Mel spectrograms, class distribution, and basic stats. Run from project root.

In [ ]:
import sys
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import yaml

# Project root: run notebook from repo root, or we detect notebooks/ parent
ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

with open(ROOT / "config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

raw_dir = ROOT / config["data"]["raw_dir"]
sr = config["data"]["sample_rate"]
feat = config["features"]
n_mels, n_fft = feat["n_mels"], feat["n_fft"]
hop_length, f_min, f_max = feat["hop_length"], feat["f_min"], feat["f_max"]
figures_dir = ROOT / "notebooks" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
print("Config loaded. raw_dir:", raw_dir)

In [ ]:
def collect_wav_paths(machine: str, label: str, top_n: int = 5):
    base = raw_dir / machine
    if not base.exists():
        return []
    paths = []
    for sub in base.iterdir():
        if sub.is_dir():
            p = sub / label
            if p.exists():
                paths.extend(sorted(p.glob("*.wav")))
    return sorted(paths)[:top_n]

samples = {}
for machine in ["fan", "slider"]:
    samples[machine] = {
        "normal": collect_wav_paths(machine, "normal", 5),
        "abnormal": collect_wav_paths(machine, "abnormal", 5),
    }
    print(machine, "normal:", len(samples[machine]["normal"]), "abnormal:", len(samples[machine]["abnormal"]))

In [ ]:
def plot_waveforms(machine: str):
    norm_paths = samples[machine]["normal"]
    abn_paths = samples[machine]["abnormal"]
    n = max(len(norm_paths), len(abn_paths))
    if n == 0:
        print(f"No data for {machine}")
        return
    fig, axes = plt.subplots(2, n, figsize=(2.5 * n, 4), squeeze=False)
    for i, (path, lab) in enumerate(zip(norm_paths + [None] * (n - len(norm_paths)), ["Normal"] * n)):
        if path is None:
            break
        y, _ = librosa.load(str(path), sr=sr, mono=True)
        axes[0, i].plot(np.arange(len(y)) / sr, y, color="C0")
        axes[0, i].set_title(f"{lab} #{i+1}")
        axes[0, i].set_xlabel("Time (s)")
    for i, (path, lab) in enumerate(zip(abn_paths + [None] * (n - len(abn_paths)), ["Abnormal"] * n)):
        if path is None:
            break
        y, _ = librosa.load(str(path), sr=sr, mono=True)
        axes[1, i].plot(np.arange(len(y)) / sr, y, color="C1")
        axes[1, i].set_title(f"{lab} #{i+1}")
        axes[1, i].set_xlabel("Time (s)")
    fig.suptitle(f"{machine.capitalize()} — Raw waveform")
    plt.tight_layout()
    out = figures_dir / f"waveform_{machine}.png"
    plt.savefig(out, dpi=120)
    plt.show()
    print("Saved:", out)

for machine in ["fan", "slider"]:
    plot_waveforms(machine)

In [ ]:
def plot_logmel(machine: str):
    norm_paths = samples[machine]["normal"]
    abn_paths = samples[machine]["abnormal"]
    n = max(len(norm_paths), len(abn_paths))
    if n == 0:
        return
    fig, axes = plt.subplots(2, n, figsize=(2.5 * n, 4), squeeze=False)
    for i, path in enumerate(norm_paths[:n]):
        y, _ = librosa.load(str(path), sr=sr, mono=True)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length, fmin=f_min, fmax=f_max)
        log_mel = librosa.power_to_db(mel, ref=np.max)
        im = axes[0, i].imshow(log_mel, aspect="auto", origin="lower", cmap="viridis")
        axes[0, i].set_title(f"Normal #{i+1}")
        axes[0, i].set_xlabel("Time (frames)")
        axes[0, i].set_ylabel("Mel")
    for i, path in enumerate(abn_paths[:n]):
        y, _ = librosa.load(str(path), sr=sr, mono=True)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length, fmin=f_min, fmax=f_max)
        log_mel = librosa.power_to_db(mel, ref=np.max)
        axes[1, i].imshow(log_mel, aspect="auto", origin="lower", cmap="viridis")
        axes[1, i].set_title(f"Abnormal #{i+1}")
        axes[1, i].set_xlabel("Time (frames)")
        axes[1, i].set_ylabel("Mel")
    fig.suptitle(f"{machine.capitalize()} — Log-Mel spectrogram")
    plt.tight_layout()
    out = figures_dir / f"logmel_{machine}.png"
    plt.savefig(out, dpi=120)
    plt.show()
    print("Saved:", out)

for machine in ["fan", "slider"]:
    plot_logmel(machine)

In [ ]:
counts = {"fan_normal": 0, "fan_abnormal": 0, "slider_normal": 0, "slider_abnormal": 0}
for machine in ["fan", "slider"]:
    base = raw_dir / machine
    if not base.exists():
        continue
    for sub in base.rglob("*.wav"):
        if "normal" in sub.parts:
            counts[f"{machine}_normal"] += 1
        elif "abnormal" in sub.parts:
            counts[f"{machine}_abnormal"] += 1

fig, ax = plt.subplots(figsize=(6, 4))
labels = list(counts.keys())
vals = [counts[k] for k in labels]
ax.bar(labels, vals, color=["C0", "C1", "C0", "C1"])
ax.set_ylabel("Number of clips")
ax.set_title("Class distribution (fan + slider)")
plt.xticks(rotation=15)
plt.tight_layout()
out = figures_dir / "class_distribution.png"
plt.savefig(out, dpi=120)
plt.show()
print("Saved:", out)

In [ ]:
stats = []
for machine in ["fan", "slider"]:
    for label in ["normal", "abnormal"]:
        base = raw_dir / machine
        paths = list(base.rglob("*.wav")) if base.exists() else []
        paths = [p for p in paths if label in p.parts]
        durs, amps_min, amps_max = [], [], []
        for p in paths:
            y, sr_here = librosa.load(str(p), sr=None, mono=True)
            durs.append(len(y) / sr_here)
            amps_min.append(float(np.min(np.abs(y))))
            amps_max.append(float(np.max(np.abs(y))))
        if durs:
            stats.append({
                "machine": machine,
                "class": label,
                "n_clips": len(durs),
                "mean_duration_s": np.mean(durs),
                "sample_rate": sr_here,
                "min_amplitude": np.min(amps_min),
                "max_amplitude": np.max(amps_max),
            })

print("Mean duration (s), sample rate, min/max amplitude per class:")
for s in stats:
    print(s)

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis("off")
table = [[s["machine"], s["class"], f"{s['mean_duration_s']:.2f}", str(s["sample_rate"]), f"{s['min_amplitude']:.4f}", f"{s['max_amplitude']:.4f}"] for s in stats]
ax.table(loc="center", colLabels=["Machine", "Class", "Mean dur (s)", "SR", "Min amp", "Max amp"], cellText=table)
out = figures_dir / "stats_table.png"
plt.savefig(out, dpi=120)
plt.show()
print("Saved:", out)